# Module 7: Walk-Forward Feature Selection with Purged CV

## Learning Objectives

By completing this notebook, you will:
1. Implement a purged walk-forward CV splitter from scratch
2. Apply it to real time series data for walk-forward feature selection
3. Track which features are selected at each rebalance point (selection heatmap)
4. Compute feature stability metrics: selection frequency and Kendall's W
5. Demonstrate the leakage problem by comparing purged vs naive selection

## Prerequisites
- Module 07 Guide 02: Temporal validation concepts
- Notebook 01: Granger feature ranking (optional but recommended)

## Estimated Time: 15 minutes

## Key Reference
de Prado, M.L. (2018). *Advances in Financial Machine Learning*. Wiley. Chapter 7.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from scipy.stats import kendalltau, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
print('Setup complete.')

### 1.1 Generate Realistic Time Series Data

We generate a multivariate time series with:
- A target with AR(2) dynamics and regime-varying noise
- 25 candidate features: 8 truly predictive, 17 irrelevant
- Overlapping rolling-window features to demonstrate the leakage problem

In [ ]:
# Purpose: Generate realistic time series with known predictive structure
# Key: We know which features are truly predictive so we can evaluate methods

T = 600   # 50 years of monthly data
N_FEATURES = 25
N_CAUSAL = 8   # first 8 features truly predict target
ROLL_WINDOW = 12  # rolling window for feature construction (creates label overlap)

dates = pd.date_range('1975-01-31', periods=T, freq='ME')
rng = np.random.default_rng(42)

# --- Target: industrial production growth (AR(2) + regime noise) ---
ip_growth = np.zeros(T)
ip_growth[0] = rng.normal(0, 0.5)
ip_growth[1] = 0.4 * ip_growth[0] + rng.normal(0, 0.5)
for t in range(2, T):
    regime_vol = 1.5 if t % 100 < 20 else 0.5  # crisis periods
    ip_growth[t] = 0.35 * ip_growth[t-1] - 0.15 * ip_growth[t-2] + rng.normal(0, regime_vol)

target = pd.Series(ip_growth, index=dates, name='IP_growth')

# --- Raw features: AR(1) processes ---
raw_features = np.zeros((T, N_FEATURES))

for j in range(N_FEATURES):
    phi = rng.uniform(0.3, 0.8)  # autocorrelation
    raw_features[0, j] = rng.normal(0, 1)
    for t in range(1, T):
        raw_features[t, j] = phi * raw_features[t-1, j] + rng.normal(0, 1)

# Add predictive relationship: causal features lead target by 1 period
for j in range(N_CAUSAL):
    strength = rng.uniform(0.15, 0.35)
    raw_features[1:, j] += strength * ip_growth[:-1]

feature_names = [f'Feature_{j+1:02d}' for j in range(N_FEATURES)]
raw_df = pd.DataFrame(raw_features, index=dates, columns=feature_names)

# --- Construct rolling-window features (creates label overlap) ---
# Rolling 12-month mean — observation at t uses data from [t-11, t]
rolling_features = raw_df.rolling(window=ROLL_WINDOW, min_periods=ROLL_WINDOW).mean()
rolling_features.columns = [f'Roll_{c}' for c in raw_df.columns]

# Combined feature set
features = pd.concat([raw_df, rolling_features], axis=1).dropna()
target_aligned = target.loc[features.index]

print(f'Dataset: {len(target_aligned)} observations, {features.shape[1]} features')
print(f'True causal features (raw): Feature_01 through Feature_{N_CAUSAL:02d}')
print(f'Rolling window: {ROLL_WINDOW} months (creates label overlap between observations)')
print(f'Date range: {dates[ROLL_WINDOW-1].date()} to {dates[-1].date()}')

---

## 2. Implementing the Purged Walk-Forward Splitter

The key component from de Prado (2018). Purging removes training observations
whose label windows overlap with the test period.

In [ ]:
# Purpose: Implement PurgedWalkForwardSplitter from scratch
# This is the production-grade temporal validation tool

class PurgedWalkForwardSplitter:
    """
    Walk-forward cross-validation with purging and embargo.

    Based on de Prado (2018) Chapter 7.
    Purging removes training observations whose label spans overlap
    with the test period. Embargo removes observations immediately
    after the test period.

    Parameters
    ----------
    n_splits : int
        Number of walk-forward folds.
    test_size : int
        Number of observations in each test fold.
    embargo_pct : float
        Embargo size as fraction of total observations.
    purge : bool
        If False, skip purging (naive walk-forward for comparison).
    """

    def __init__(
        self,
        n_splits: int = 5,
        test_size: int = None,
        embargo_pct: float = 0.01,
        purge: bool = True,
    ):
        self.n_splits = n_splits
        self.test_size = test_size
        self.embargo_pct = embargo_pct
        self.purge = purge

    def split(self, X, y=None, label_ends=None):
        """
        Generate (train_indices, test_indices) pairs.

        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
        y : ignored
        label_ends : array-like of int, optional
            label_ends[i] = last index used to construct observation i's label.
            For rolling-window features: label_ends[i] = i (end of window = current obs).
            For overlapping labels: label_ends[i] = i + overlap.
            If None, assumes point-in-time labels (no overlap).
        """
        n = len(X)
        test_size = self.test_size or max(1, n // (self.n_splits + 1))
        embargo_size = max(1, int(n * self.embargo_pct))

        first_test_start = n - test_size * self.n_splits
        if first_test_start < test_size:
            raise ValueError(
                f'Insufficient data for {self.n_splits} splits with test_size={test_size}'
            )

        for fold in range(self.n_splits):
            test_start = first_test_start + fold * test_size
            test_end = min(test_start + test_size, n)
            test_idx = np.arange(test_start, test_end)

            # All observations before the test period are training candidates
            train_candidates = np.arange(0, test_start)

            if self.purge and label_ends is not None:
                # Purge: remove training obs whose label span reaches into test period
                # An observation i is purged if its label end index >= test_start
                label_end_arr = np.asarray(label_ends)
                purge_mask = label_end_arr[train_candidates] < test_start
                train_idx = train_candidates[purge_mask]

                # Embargo: also remove observations after test period (if they
                # would overlap with next fold's training due to autocorrelation)
                # Here we apply embargo to the current training set by removing
                # obs that are within embargo_size of the test start
                embargo_zone = np.arange(
                    max(0, test_start - embargo_size), test_start
                )
                train_idx = train_idx[
                    ~np.isin(train_idx, embargo_zone)
                ]
            else:
                # No purging — standard walk-forward
                train_idx = train_candidates

            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


# Demonstrate the splitter
N_SPLITS = 5
TEST_SIZE = 48  # 4 years

# Label ends for rolling-window features:
# Rolling 12-month mean at position i uses data [i-11, i], so label_end = i
# For simplicity in this demonstration, label_end[i] = i + ROLL_WINDOW - 1
# (the overlap extends ROLL_WINDOW - 1 periods forward)
label_ends_roll = np.arange(ROLL_WINDOW - 1, len(features) + ROLL_WINDOW - 1)
label_ends_roll = np.clip(label_ends_roll, 0, len(features) - 1)

print('Purged Walk-Forward Splitter Demo')
print('='*55)
purged_splitter = PurgedWalkForwardSplitter(
    n_splits=N_SPLITS, test_size=TEST_SIZE, embargo_pct=0.02, purge=True
)

for fold, (train_idx, test_idx) in enumerate(
    purged_splitter.split(features.values, label_ends=label_ends_roll)
):
    print(f'Fold {fold+1}:  Train [{features.index[train_idx[0]].year}-{features.index[train_idx[-1]].year}] '
          f'({len(train_idx)} obs)  '
          f'Test [{features.index[test_idx[0]].year}-{features.index[test_idx[-1]].year}] '
          f'({len(test_idx)} obs)')

---

## 3. Walk-Forward Feature Selection

Re-select features at each rebalance point using only historically available data.
We compare two methods: mutual information (fast, good for exploration) and
LASSO coefficients (embeds selection in regularised regression).

In [ ]:
# Purpose: Walk-forward feature selection with purged CV
# Track which features are selected at each rebalance point

TOP_K = 10  # Select top-k features per fold

def select_features_at_fold(
    target: pd.Series,
    features: pd.DataFrame,
    train_idx: np.ndarray,
    top_k: int = 10,
    method: str = 'lasso',
) -> dict:
    """
    Select top-k features using only training fold data.

    Returns dict with selected feature names and scores.
    """
    X_train = features.iloc[train_idx]
    y_train = target.iloc[train_idx]

    # Standardise within fold (avoids scale leakage)
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=features.columns,
        index=X_train.index,
    )

    if method == 'mutual_info':
        scores = mutual_info_regression(X_scaled.values, y_train.values, random_state=42)
        score_series = pd.Series(scores, index=features.columns)

    elif method == 'lasso':
        # LASSO with cross-validated alpha — use lagged features to avoid contemporaneous leakage
        X_lag = X_scaled.shift(1).dropna()
        y_aligned = y_train.loc[X_lag.index]
        lasso = Lasso(alpha=0.05, max_iter=2000, random_state=42)
        lasso.fit(X_lag.values, y_aligned.values)
        score_series = pd.Series(np.abs(lasso.coef_), index=features.columns)

    # Select top-k by score
    top_features = score_series.nlargest(top_k).index.tolist()

    return {
        'selected': top_features,
        'scores': score_series,
        'selection_binary': {col: col in top_features for col in features.columns},
    }


# Run purged walk-forward selection
print('Running purged walk-forward feature selection (LASSO)...')
purged_selections = []
purged_scores_list = []
fold_dates = []

for fold, (train_idx, test_idx) in enumerate(
    purged_splitter.split(features.values, label_ends=label_ends_roll)
):
    result = select_features_at_fold(
        target_aligned, features, train_idx, top_k=TOP_K, method='lasso'
    )
    purged_selections.append(result['selection_binary'])
    purged_scores_list.append(result['scores'])

    fold_end_date = features.index[train_idx[-1]]
    fold_dates.append(fold_end_date)

    print(f'Fold {fold+1} (train through {fold_end_date.date()}): '
          f'selected {result["selected"][:5]}...')

selection_matrix = pd.DataFrame(purged_selections, index=fold_dates).astype(int)
scores_matrix = pd.DataFrame(purged_scores_list, index=fold_dates)

print(f'\nSelection matrix shape: {selection_matrix.shape}')
print(f'(rows=folds, columns=features, values=selected/not)')

---

## 4. Feature Stability Metrics

Compute selection frequency and Kendall's W to identify stable vs unstable features.

In [ ]:
# Purpose: Compute feature stability across walk-forward folds
# Key: Stable features (high selection frequency) are more trustworthy

# Selection frequency: proportion of folds where each feature is selected
sel_frequency = selection_matrix.mean(axis=0).sort_values(ascending=False)

# Pairwise Spearman correlation of feature scores across folds
spearman_corrs = []
n_folds = len(scores_matrix)
for i in range(n_folds):
    for j in range(i + 1, n_folds):
        rho, _ = spearmanr(scores_matrix.iloc[i], scores_matrix.iloc[j])
        spearman_corrs.append(rho)
mean_spearman = np.mean(spearman_corrs)

# Kendall's W (concordance)
rank_matrix = scores_matrix.rank(axis=1)
m = n_folds
n_feat = features.shape[1]
R = rank_matrix.sum(axis=0)  # sum of ranks for each feature
R_bar = R.mean()
S = ((R - R_bar) ** 2).sum()
kendalls_W = 12 * S / (m ** 2 * (n_feat ** 3 - n_feat))

print('Feature Stability Analysis')
print('='*50)
print(f'Number of folds: {n_folds}')
print(f"Kendall's W (ranking concordance): {kendalls_W:.3f}")
print(f'  (1=perfect agreement, 0=no agreement)')
print(f'Mean pairwise Spearman rank correlation: {mean_spearman:.3f}')

print(f'\nTop 10 most stably selected features:')
for feat, freq in sel_frequency.head(10).items():
    true_causal = feat.startswith('Feature_') and int(feat.split('_')[1]) <= N_CAUSAL
    marker = '[TRUE]' if true_causal else '[NOI] '
    bar = '█' * int(freq * 20) + '░' * (20 - int(freq * 20))
    print(f'  {marker} {feat:20s} {bar} {freq:.0%}')

print(f'\nBottom 5 least stably selected:')
for feat, freq in sel_frequency.tail(5).items():
    print(f'  {feat}: {freq:.0%}')

In [ ]:
# Purpose: Visualise selection heatmap — features vs folds
# Key: Rows = features, columns = folds; dark = selected

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Heatmap: selection across folds ---
# Sort features by selection frequency for readability
sorted_feats = sel_frequency.index.tolist()
heatmap_data = selection_matrix[sorted_feats].T  # features x folds

# Colour code: light=not selected, dark blue=selected; red outline for true causal
sns.heatmap(
    heatmap_data.astype(float),
    ax=axes[0],
    cmap='Blues',
    linewidths=0.3,
    linecolor='white',
    xticklabels=[d.strftime('%Y-%m') for d in fold_dates],
    yticklabels=sorted_feats,
    cbar_kws={'label': 'Selected (1=yes)'},
    vmin=0, vmax=1,
)

# Highlight true causal features with yellow text
ax_yticks = axes[0].get_yticklabels()
for tick in ax_yticks:
    feat_name = tick.get_text()
    is_causal = feat_name.startswith('Feature_') and int(feat_name.split('_')[1]) <= N_CAUSAL
    tick.set_color('#C0392B' if is_causal else 'black')
    tick.set_fontweight('bold' if is_causal else 'normal')
    tick.set_fontsize(7)

axes[0].set_title('Feature Selection Heatmap\n(red labels = truly causal features)', fontsize=11)
axes[0].set_xlabel('Fold rebalance date')
axes[0].set_ylabel('Feature (sorted by selection frequency)')
axes[0].tick_params(axis='x', rotation=30)

# --- Bar chart: selection frequency ---
colors = ['#C0392B' if (f.startswith('Feature_') and int(f.split('_')[1]) <= N_CAUSAL)
          else '#2980B9'
          for f in sorted_feats]
axes[1].barh(range(len(sorted_feats)), sel_frequency[sorted_feats].values,
             color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_yticks(range(len(sorted_feats)))
axes[1].set_yticklabels(sorted_feats, fontsize=7)
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1, label='50% threshold')
axes[1].set_xlabel('Selection Frequency')
axes[1].set_title('Feature Selection Frequency\n(red=truly causal)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig('walk_forward_selection_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: walk_forward_selection_heatmap.png')

---

## 5. The Leakage Problem: Purged vs Naive Selection

This is the core demonstration of why purging matters. We compare:
- **Naive walk-forward:** no purging, rolling-window features create label overlap
- **Purged walk-forward:** removes overlapping training observations

The prediction error differential reveals the magnitude of leakage bias.

In [ ]:
# Purpose: Compare purged vs naive walk-forward selection and prediction
# Key insight: naive selection picks different features and has optimistic CV scores

# Naive splitter: no purging
naive_splitter = PurgedWalkForwardSplitter(
    n_splits=N_SPLITS, test_size=TEST_SIZE, embargo_pct=0.0, purge=False
)

def evaluate_selection_method(
    splitter,
    target,
    features,
    top_k=10,
    label_ends=None,
    method_name='purged',
):
    """
    Walk-forward evaluation: select features and predict on test fold.
    Returns dict with per-fold results.
    """
    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(
        splitter.split(features.values, label_ends=label_ends)
    ):
        # Feature selection (within-fold)
        selection = select_features_at_fold(
            target, features, train_idx, top_k=top_k, method='lasso'
        )
        selected = selection['selected']

        # Evaluate prediction on test fold
        X_train = features.iloc[train_idx][selected]
        y_train = target.iloc[train_idx]
        X_test = features.iloc[test_idx][selected]
        y_test = target.iloc[test_idx]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        model = Ridge(alpha=1.0)
        model.fit(X_train_s, y_train.values)
        preds = model.predict(X_test_s)

        test_mse = mean_squared_error(y_test.values, preds)

        # Also record within-fold CV estimate (what you'd see without test set)
        from sklearn.model_selection import cross_val_score
        cv_scores = cross_val_score(
            Ridge(alpha=1.0), X_train_s, y_train.values,
            cv=5, scoring='neg_mean_squared_error'
        )
        cv_mse = -cv_scores.mean()

        # How many true causal features selected?
        n_true = sum(1 for f in selected if f.startswith('Feature_') and int(f.split('_')[1]) <= N_CAUSAL)

        fold_results.append({
            'fold': fold + 1,
            'method': method_name,
            'n_train': len(train_idx),
            'n_test': len(test_idx),
            'selected': selected,
            'n_true_causal': n_true,
            'cv_mse': cv_mse,
            'test_mse': test_mse,
            'optimism': cv_mse - test_mse,  # negative = CV overoptimistic
        })

    return pd.DataFrame(fold_results)


print('Evaluating purged walk-forward selection...')
purged_results = evaluate_selection_method(
    purged_splitter, target_aligned, features,
    top_k=TOP_K, label_ends=label_ends_roll, method_name='Purged'
)

print('Evaluating naive walk-forward selection...')
naive_results = evaluate_selection_method(
    naive_splitter, target_aligned, features,
    top_k=TOP_K, label_ends=None, method_name='Naive'
)

all_results = pd.concat([purged_results, naive_results], ignore_index=True)

# Summary
print('\nResults Summary')
print('='*65)
for method, grp in all_results.groupby('method'):
    print(f'\n{method} Walk-Forward:')
    print(f'  Avg true causal features selected: {grp["n_true_causal"].mean():.1f} / {N_CAUSAL}')
    print(f'  CV-estimated MSE:  {grp["cv_mse"].mean():.4f}')
    print(f'  Test MSE (actual): {grp["test_mse"].mean():.4f}')
    print(f'  CV optimism:       {grp["optimism"].mean():.4f}  (neg = CV too optimistic)')

print('\nKey finding: Naive selection shows more CV optimism (larger gap between CV and test MSE)')
print('Purged selection provides more accurate CV estimates of test performance.')

In [ ]:
# Purpose: Visualise the leakage bias across folds
# Key: The gap between CV and test MSE is the leakage bias

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel 1: CV vs Test MSE per method ---
fold_nums = purged_results['fold'].values
x = np.arange(len(fold_nums))
width = 0.35

axes[0].bar(x - width/2, purged_results['cv_mse'], width, label='Purged CV-MSE',
            color='#2ECC71', alpha=0.8)
axes[0].bar(x + width/2, naive_results['cv_mse'], width, label='Naive CV-MSE',
            color='#E74C3C', alpha=0.8)
axes[0].plot(x - width/2, purged_results['test_mse'], 'go-', markersize=8,
             label='Purged Test MSE', linewidth=2)
axes[0].plot(x + width/2, naive_results['test_mse'], 'rs-', markersize=8,
             label='Naive Test MSE', linewidth=2)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Fold {i+1}' for i in range(len(fold_nums))])
axes[0].set_ylabel('MSE')
axes[0].set_title('CV vs True Test MSE\n(gap = leakage bias)', fontsize=11)
axes[0].legend(fontsize=8)

# --- Panel 2: True causal features selected ---
axes[1].plot(fold_nums, purged_results['n_true_causal'], 'go-', linewidth=2,
             markersize=8, label=f'Purged (max={N_CAUSAL})')
axes[1].plot(fold_nums, naive_results['n_true_causal'], 'rs-', linewidth=2,
             markersize=8, label='Naive')
axes[1].axhline(N_CAUSAL, color='black', linestyle='--', alpha=0.5, label=f'Max causal={N_CAUSAL}')
axes[1].axhline(TOP_K * N_CAUSAL / features.shape[1], color='gray', linestyle=':',
                label='Random baseline')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('True causal features selected')
axes[1].set_title('Feature Selection Quality\n(higher = better)', fontsize=11)
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, N_CAUSAL + 1)

# --- Panel 3: Summary comparison ---
metrics = ['CV-MSE\n(estimated)', 'Test MSE\n(true)', 'True causal\nselected']
purged_vals = [
    purged_results['cv_mse'].mean(),
    purged_results['test_mse'].mean(),
    purged_results['n_true_causal'].mean(),
]
naive_vals = [
    naive_results['cv_mse'].mean(),
    naive_results['test_mse'].mean(),
    naive_results['n_true_causal'].mean(),
]

x3 = np.arange(3)
# Normalise each metric for comparison
max_vals = [max(p, n) for p, n in zip(purged_vals, naive_vals)]
purged_norm = [p/m for p, m in zip(purged_vals, max_vals)]
naive_norm = [n/m for n, m in zip(naive_vals, max_vals)]

axes[2].bar(x3 - 0.2, purged_norm, 0.4, label='Purged', color='#2ECC71', alpha=0.8)
axes[2].bar(x3 + 0.2, naive_norm, 0.4, label='Naive', color='#E74C3C', alpha=0.8)
axes[2].set_xticks(x3)
axes[2].set_xticklabels(metrics, fontsize=9)
axes[2].set_ylabel('Relative to method maximum')
axes[2].set_title('Summary Comparison\n(lower MSE = better; higher causal = better)', fontsize=11)
axes[2].legend()

for i, (pv, nv) in enumerate(zip(purged_vals, naive_vals)):
    axes[2].text(i - 0.2, purged_norm[i] + 0.02, f'{pv:.3f}', ha='center', fontsize=8)
    axes[2].text(i + 0.2, naive_norm[i] + 0.02, f'{nv:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('purged_vs_naive_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: purged_vs_naive_comparison.png')

---

## Exercise 7.2: Implement Combinatorial Purged CV

**Task:** Extend `PurgedWalkForwardSplitter` to generate all $\binom{N}{k}$ combinations
of $k$ test blocks from $N$ total data blocks, yielding a distributional estimate
of out-of-sample performance.

**Requirements:**
1. Divide the data into $N=6$ equal-size blocks
2. Generate all $\binom{6}{2} = 15$ combinations of 2 test blocks
3. For each combination: build train set from remaining 4 blocks, apply purging
4. Collect test MSE across all 15 combinations and plot the distribution

**Hint:** Use `from itertools import combinations`.

In [ ]:
# YOUR CODE HERE
# ---------------
from itertools import combinations

N_BLOCKS = 6
K_TEST = 2

# Step 1: Divide data into N_BLOCKS equal blocks
n_total = len(features)
block_size = n_total // N_BLOCKS
blocks = []
for i in range(N_BLOCKS):
    start = i * block_size
    end = (i + 1) * block_size if i < N_BLOCKS - 1 else n_total
    blocks.append(np.arange(start, end))

print(f'Data divided into {N_BLOCKS} blocks of ~{block_size} observations each')
print(f'Number of test combinations: C({N_BLOCKS},{K_TEST}) = {len(list(combinations(range(N_BLOCKS), K_TEST)))}')

# TODO: Implement the combinatorial purged CV loop
# For each (i, j) combination of test blocks:
#   test_idx = concatenate blocks[i] and blocks[j]
#   train_blocks = all blocks except i and j
#   train_idx = concatenate train_blocks (with purging)
#   select features and evaluate MSE

cpcv_mses = []  # collect test MSE from each combination

# TODO: Fill in the loop
print('\nTODO: Implement the combinatorial CV loop')
print('Expected: 15 test MSE values to plot as a distribution')

In [ ]:
# AUTO-GRADED TEST
def test_exercise_72():
    assert len(blocks) == N_BLOCKS, f'Expected {N_BLOCKS} blocks, got {len(blocks)}'
    expected_combos = len(list(combinations(range(N_BLOCKS), K_TEST)))
    assert expected_combos == 15, f'Expected 15 combinations, got {expected_combos}'
    print(f'Exercise 7.2: {N_BLOCKS} blocks and {expected_combos} combinations correctly set up.')
    print('Complete the combinatorial CV loop to finish this exercise.')

test_exercise_72()

---

## Summary

### Key Takeaways

1. **Standard k-fold CV creates look-ahead bias** for time series — observations from the future train models evaluated on the past.

2. **Purging removes training observations** whose label windows overlap with the test period — essential for rolling-window features (e.g., 12-month rolling means).

3. **Selection frequency** (fraction of folds where a feature is selected) is a robust stability metric — features selected > 70% of folds are reliable.

4. **Kendall's W** measures ranking concordance across folds — W > 0.5 indicates stable feature ordering.

5. **Naive vs purged comparison** demonstrates that purging reduces optimism bias in CV performance estimates and selects more truly predictive features.

### What's Next

Notebook 03 covers regime-aware feature selection — adapting feature sets to different market conditions using HMM-detected regimes.

### References

- de Prado, M.L. (2018). *Advances in Financial Machine Learning*. Wiley. Chapter 7.
- Bergmeir, C. & Benítez, J.M. (2012). On the use of cross-validation for time series predictor evaluation. *Information Sciences*, 191, 192–213.